# Bitcoin Price Close Prediction

Uses all available daily close data (up to yesterday) to predict today's close.

In [ ]:
import csv
from datetime import datetime

import llb
import numpy as np

CSV_PATH = "btc_price.csv"
# ── Load CSV and keep one close per day (latest timestamp in each day) ────────
records = []
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    for r in reader:
        ts = datetime.fromisoformat(r["timeOpen"].replace("Z", "+00:00")).replace(tzinfo=None)
        close = float(str(r["close"]).replace(",", ""))
        records.append((ts, close))

if not records:
    raise ValueError("CSV has no rows.")

latest_per_day = {}
for ts, close in records:
    d = ts.date()
    prev = latest_per_day.get(d)
    if prev is None or ts > prev[0]:
        latest_per_day[d] = (ts, close)

daily_rows = sorted(latest_per_day.values(), key=lambda x: x[0])
if len(daily_rows) < 10:
    raise ValueError(f"Need at least 10 daily rows, got {len(daily_rows)}.")

# Predict the last available day using all prior daily history.
today_date, today_close = daily_rows[-1]
train_rows = daily_rows[:-1]

btc_closes = np.asarray([c for _, c in train_rows], dtype=float)
num_days = int(len(btc_closes))
if num_days < 5:
    raise ValueError("Not enough close-price observations for inference.")

# Basic parsing sanity check: BTC close should not be tiny.
if float(np.nanmax(btc_closes)) < 1000.0:
    raise ValueError(
        "Parsed BTC closes look too small (<1000). Check delimiter/columns in CSV parsing."
    )

text = (
    "I have all available daily Bitcoin close prices up to yesterday. "
    "Model latent level and volatility, then predict the next-day close price."
)
data = {
    "num_days": num_days,
    "btc_closes": btc_closes.tolist(),
}
targets = ["future_btc_close"]

posterior = llb.infer(
    text=text,
    data=data,
    targets=targets,
    api_url="https://api.openai.com/v1/chat/completions",
    api_key="OPEN_AI_API_KEY",
    api_model="gpt-4.1-mini",
    n_models=15,
    mcmc_num_warmup=50,
    mcmc_num_samples=100,
)

pred_arr = np.asarray(posterior["future_btc_close"], dtype=float)
pred_samples = pred_arr if pred_arr.ndim == 1 else pred_arr[:, 0]
pred_close = float(np.mean(pred_samples))

err = pred_close - today_close
abs_err = abs(err)
pct_err = (abs_err / today_close) * 100.0

print(f"Raw rows loaded         : {len(records)}")
print(f"Unique daily rows       : {len(daily_rows)}")
print(f"Train range             : {train_rows[0][0].date()} -> {train_rows[-1][0].date()} ({len(train_rows)} days)")
print(f"Predict day             : {today_date.date()}")
print(f"Actual close            : {today_close:,.2f}")
print(f"Predicted close         : {pred_close:,.2f}")
print(f"Error                   : {err:+,.2f} (abs={abs_err:,.2f}, pct={pct_err:.2f}%)")


Number of requested models: 15
Number of generated models: 15
Number of deduplicated models: 0
Number of invalid models (syntax/parsing): 0
Number of generation request failures (timeout/network/API): 0
Number of models missing required targets: 0
Number of models that failed to compile: 0
Number of models that failed during inference: 1
Number of models dropped due to target shape mismatch: 0
Number of models dropped due to non-finite log bound: 0
Number of valid models used in final aggregation: 14
--- Model Averaging Summary ---
Target: future_btc_close
Number of models: 14
flat mean prediction: 38459.448954
weighted mean prediction: 72250.637109
Difference (weighted - flat): 33791.188155

Top 5 models by weight for target 'future_btc_close':
model=2, weight=1.000000, mu_i=72250.637109
model=1, weight=0.000000, mu_i=66876.334727
model=4, weight=0.000000, mu_i=69903.706836
model=10, weight=0.000000, mu_i=70239.099141
model=0, weight=0.000000, mu_i=56043.832787
2 least-weighted models